In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
columns_to_label_encode = ['NOMBRE_UNIDAD_EDUC', 'NOMBRE_COMUNA_EGRESO',
                            'PAGO_MATRICULA2020', 'PAGO_MENSUAL2020', 'PAGO_MATRICULA2021', 'PAGO_MENSUAL2021',
                            'PAGO_MATRICULA2022', 'PAGO_MENSUAL2022', 'PAGO_MATRICULA2023', 'PAGO_MENSUAL2023']

label_encoder = LabelEncoder()
for column in columns_to_label_encode:
    df.loc[:, column] = label_encoder.fit_transform(df.loc[:, column])

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_normalized = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

In [ ]:
data = df_normalized.drop(['MRUN', 'CLEC_REG_ACTUAL', 'MATE1_REG_ACTUAL', 'MATE2_REG_ACTUAL', 'PROMEDIO_CM_MAX', ], axis=1).values
columns = df_normalized.drop(['MRUN', 'CLEC_REG_ACTUAL', 'MATE1_REG_ACTUAL', 'MATE2_REG_ACTUAL', 'PROMEDIO_CM_MAX', ], axis=1).columns
target = df_normalized['PROMEDIO_CM_MAX'].values

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2)

### Variaci?n de hiperpar?metros

In [ ]:
# Define the hyperparameter grid
param_grid = {
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50,100, 200],
    'min_samples_split': [2, 5, 10],
    'max_depth': [None, 10,20],
    'max_features': ['sqrt', 'log2', None],
    'oob_score': [True]
}
# Create a random forest regressor
rf = RandomForestRegressor()

# Perform grid search with cross-validation
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(rf, param_grid, cv=3,verbose=3)
grid_search.fit(X_train[:5000], y_train[:5000])

# Get the best hyperparameters
best_params = grid_search.best_params_



In [ ]:
# Train the model with the best hyperparameters
rf_best = RandomForestRegressor(**best_params)
rf_best.fit(X_train, y_train)

# Evaluate the model on the test set
predictions = rf_best.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f'Testing Mean Squared Error: {mse}')
mae = mean_absolute_error(y_test, predictions)
print(f'Testing Mean Absolute Error: {mae}')
r2 = r2_score(y_test, predictions)
print(f'Test R-squared: {r2}')
print(best_params)

In [ ]:
importance=np.array(rf_best.feature_importances_)

In [ ]:
from sklearn.inspection import permutation_importance
# Calcular la importancia de las caracter?sticas
results = permutation_importance(rf_best, X_test, y_test, n_repeats=10, random_state=0)

# Obtener la media de la disminuci?n en el score de las predicciones para cada caracter?stica
importances = results.importances_mean

# Crear un DataFrame para visualizar mejor la importancia de cada caracter?stica
feature_importances = pd.DataFrame({'feature': columns, 'importance': importances})
feature_importances = feature_importances.sort_values(by='importance', ascending=False)

# Mostrar las caracter?sticas m?s importantes
print(feature_importances)

# Visualizar la importancia de las caracter?sticas
plt.figure(figsize=(12, 8))
plt.barh(feature_importances['feature'][:5], feature_importances['importance'][:5])
plt.xlabel('Importancia de Caracter?sticas')
plt.ylabel('Caracter?sticas')
plt.title('Importancia de Caracter?sticas basada en Permutaciones')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Plot the training and validation loss for rf_best
import seaborn as sns
loss_train = []
loss_val = []
for i in range(len(rf_best.estimators_)):
    loss_train.append(rf_best.estimators_[i].tree_.impurity.sum())

plt.figure(figsize=(10, 6))
sns.histplot(loss_train,kde=True)
plt.xlabel('P?rdida')

plt.ylabel('Cantidad de ?rboles')
plt.legend()
plt.show()


In [ ]:
# prompt: create a dict between columns and importance, ordering on a descendent value of importance

feature_importances = dict(zip(columns, importance))
sorted_features = sorted(feature_importances.items(), key=lambda x: x[1], reverse=True)
print(sorted_features)


In [ ]:
for a in range(len(columns)):
    print(f'{columns[a]}: {importance[a]:.3f}')